# 07 - Enterprise Domain Event Simulation and Bronze

Generates deterministic, fictional enterprise master data and source-shaped events for routes, fleets, pseudonymous passengers and bookings, baggage, workforce rosters, retail/POS, turnaround phases, and customer experience.

No names, contact details, loyalty identifiers, biometrics, credentials, tenant identifiers, or operational endpoints are generated. Every record is synthetic and advisory analytics remain outside all operational control paths.

In [ ]:
import hashlib
import json
import math
import random as random_module
from datetime import datetime, timedelta

from delta.tables import DeltaTable
from pyspark.sql import Row, functions as F

config = spark.table('bronze_demo_config').first().asDict()
assert int(config['random_seed']) >= 0 and config['is_synthetic'] is True
BASE = datetime.strptime(config['base_date'], '%Y-%m-%d')
OBSERVATION_TS = config['observation_timestamp']
RNG = random_module.Random(int(config['random_seed']) + 700)
BATCH_ID = 'ENTERPRISE-' + config['base_date'].replace('-', '') + '-SEED-' + str(config['random_seed'])
PROCESSING_MODE = config.get('processing_mode', 'full_reset')
SCALE = int(config.get('scale_factor', 1))
ROUTES_PER_AIRPORT = int(config.get('routes_per_airport', 8))
EMPLOYEES_PER_AIRPORT = int(config.get('employees_per_airport', 48)) * SCALE
OUTLETS_PER_TERMINAL = int(config.get('retail_outlets_per_terminal', 4)) * SCALE
ENTERPRISE_KEYS = {
    'bronze_organization':'org_unit_id','bronze_route':'route_id','bronze_aircraft_fleet':'aircraft_instance_id',
    'bronze_work_team':'work_team_id','bronze_skill':'skill_id','bronze_shift':'shift_id',
    'bronze_employee':'employee_id','bronze_employee_skill':'employee_skill_id',
    'bronze_employee_roster':'roster_assignment_id','bronze_retail_outlet':'outlet_id',
    'bronze_retail_product':'product_id','bronze_flight_route':'flight_event_id','bronze_flight_leg':'leg_id',
    'bronze_customer':'customer_token','bronze_passenger':'passenger_token','bronze_booking':'booking_id',
    'bronze_boarding_event':'boarding_event_id','bronze_baggage_journey':'bag_token',
    'bronze_baggage_scan':'baggage_scan_id','bronze_ramp_service_task':'ramp_task_id',
    'bronze_maintenance_work_order':'work_order_id','bronze_retail_pos':'pos_event_id',
    'bronze_turnaround_phase':'phase_event_id','bronze_customer_experience':'cx_event_id',
    'bronze_recommendation_event':'recommendation_id','bronze_event_quality_cases':'test_case_id'}
SYNTHETIC_MASTER_TABLES = {
    'bronze_organization','bronze_route','bronze_aircraft_fleet','bronze_work_team','bronze_skill',
    'bronze_shift','bronze_employee','bronze_employee_skill','bronze_retail_outlet','bronze_retail_product',
    'bronze_customer','bronze_passenger'}


def stable_token(kind, *parts):
    raw = '|'.join([kind, str(config['random_seed'])] + [str(part) for part in parts])
    return hashlib.sha256(raw.encode('utf-8')).hexdigest()[:24]


def bronze_row(table_name, record_key, source_event_timestamp, **fields):
    payload_hash = hashlib.sha256(json.dumps(fields, default=str, sort_keys=True, separators=(',', ':')).encode('utf-8')).hexdigest()
    classification = 'SyntheticMaster' if table_name in SYNTHETIC_MASTER_TABLES else 'SyntheticOperational'
    return Row(
        **fields, source_event_timestamp=source_event_timestamp, ingestion_timestamp=OBSERVATION_TS,
        source_simulation_id='airport-ops-enterprise-v2', source_record_key=str(record_key),
        schema_version='2.0', payload_hash=payload_hash, batch_id=BATCH_ID, is_synthetic=True,
        data_classification=classification, source_name='DeterministicSyntheticGenerator',
        source_url='repo://notebooks/07_Generate_Enterprise_Bronze', source_as_of_date=config['base_date'],
        generator_version=config['generator_version'], random_seed=int(config['random_seed']),
        record_source='SyntheticGenerator')


def write_delta(generated_rows, table_name, chunk_size=25000):
    assert generated_rows, table_name + ' cannot be empty'
    row_dicts = [row.asDict(recursive=True) for row in generated_rows]
    columns = list(row_dicts[0])
    null_columns = [column for column in columns if all(row[column] is None for row in row_dicts)]
    typed_columns = [column for column in columns if column not in null_columns]
    schema_examples = []
    unresolved = set(typed_columns)
    for row in row_dicts:
        if any(row[column] is not None for column in unresolved):
            schema_examples.append({column: row[column] for column in typed_columns})
            unresolved = {column for column in unresolved if row[column] is None}
        if not unresolved:
            break
    assert not unresolved, table_name + ' has unresolved typed columns: ' + ','.join(sorted(unresolved))
    schema = spark.createDataFrame(schema_examples).schema
    frame = None
    for offset in range(0, len(row_dicts), chunk_size):
        chunk = [{column: row[column] for column in typed_columns} for row in row_dicts[offset:offset + chunk_size]]
        chunk_frame = spark.createDataFrame(chunk, schema=schema)
        frame = chunk_frame if frame is None else frame.unionByName(chunk_frame)
    for column in null_columns:
        frame = frame.withColumn(column, F.lit(None).cast('string'))
    if 'generated_at_utc' not in frame.columns:
        frame = frame.withColumn('generated_at_utc', F.current_timestamp())
    if PROCESSING_MODE == 'incremental' and spark.catalog.tableExists(table_name):
        spark.conf.set('spark.databricks.delta.schema.autoMerge.enabled', 'true')
        key = ENTERPRISE_KEYS[table_name]
        (DeltaTable.forName(spark, table_name).alias('target')
            .merge(frame.alias('source'), f'target.{key} = source.{key}')
            .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
        action = 'merged'
    else:
        frame.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable(table_name)
        action = 'overwrote'
    print(action, table_name, len(generated_rows))


def rows(table_name):
    return [row.asDict() for row in spark.table(table_name).collect()]


def distance_km(origin, destination):
    lat1, lon1 = math.radians(origin['latitude']), math.radians(origin['longitude'])
    lat2, lon2 = math.radians(destination['latitude']), math.radians(destination['longitude'])
    delta_lat, delta_lon = lat2 - lat1, lon2 - lon1
    value = math.sin(delta_lat / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(delta_lon / 2) ** 2
    return int(round(6371.0 * 2 * math.atan2(math.sqrt(value), math.sqrt(1 - value))))


airports = sorted(rows('bronze_airport'), key=lambda row: row['airport_id'])
airlines = sorted(rows('bronze_airline'), key=lambda row: row['airline_id'])
aircraft_types = sorted(rows('bronze_aircraft'), key=lambda row: row['aircraft_type_id'])
gates = sorted(rows('bronze_gate'), key=lambda row: row['gate_id'])
flights = sorted(rows('bronze_flight_turnaround'), key=lambda row: row['flight_event_id'])
service_assumptions = rows('bronze_airline_airport_service')
eligibility_assumptions = rows('bronze_airline_aircraft_eligibility')
assert len(airports) == int(config['airport_count']) and len(airlines) == int(config['airline_count'])
assert len(aircraft_types) == int(config['aircraft_type_count']) and gates and flights

In [ ]:
# Fictional organization, constrained routes/fleet, teams, skills, workforce, rosters, and retail outlets.
regions = sorted({airport['region'] for airport in airports})
organization_rows = [bronze_row(
    'bronze_organization', 'ORG-HQ', BASE, org_unit_id='ORG-HQ', parent_org_unit_id=None,
    org_unit_type='CorporateHeadquarters', org_unit_name='Asteria Airports Group HQ',
    operating_region='Northstar', fictional_relationship_flag=True)]
for index, region in enumerate(regions, start=1):
    organization_rows.append(bronze_row(
        'bronze_organization', 'ORG-REG-' + str(index), BASE,
        org_unit_id='ORG-REG-' + str(index), parent_org_unit_id='ORG-HQ', org_unit_type='OperatingRegion',
        org_unit_name='Fictional ' + region + ' Operating Region', operating_region=region,
        fictional_relationship_flag=True))

services_by_airport = {}
for service in service_assumptions:
    services_by_airport.setdefault(service['airport_id'], []).append(service['airline_id'])
airline_by_id = {airline['airline_id']: airline for airline in airlines}
route_rows = []
for origin_index, origin in enumerate(airports):
    destinations = [airport for airport in airports if airport['airport_id'] != origin['airport_id']]
    served_airline_ids = sorted(services_by_airport[origin['airport_id']])
    assert len(served_airline_ids) >= ROUTES_PER_AIRPORT
    for route_index in range(ROUTES_PER_AIRPORT):
        destination = destinations[(origin_index * 3 + route_index) % len(destinations)]
        airline = airline_by_id[served_airline_ids[route_index % len(served_airline_ids)]]
        route_id = 'R-' + origin['airport_id'] + '-' + str(route_index + 1).zfill(2)
        route_rows.append(bronze_row(
            'bronze_route', route_id, BASE, route_id=route_id, origin_airport_id=origin['airport_id'],
            destination_airport_id=destination['airport_id'], airline_id=airline['airline_id'],
            route_category='InterRegion' if origin['region'] != destination['region'] else 'IntraRegion',
            distance_km=distance_km(origin, destination), scheduled_frequency_daily=1 + route_index % 4,
            service_assumption_id='SERVICE-' + origin['airport_id'] + '-' + airline['airline_id']))

service_airports_by_airline = {}
for assumption in service_assumptions:
    service_airports_by_airline.setdefault(assumption['airline_id'], []).append(assumption['airport_id'])
fleet_rows = []
fleet_number = 0
for assumption in sorted(eligibility_assumptions, key=lambda item: (item['airline_id'], item['aircraft_type_id'])):
    for instance_index in range(SCALE):
        fleet_number += 1
        airline_id = assumption['airline_id']
        aircraft_type_id = assumption['aircraft_type_id']
        base_airports = sorted(service_airports_by_airline[airline_id])
        base_airport_id = base_airports[(fleet_number + instance_index) % len(base_airports)]
        aircraft_instance_id = 'FLEET-' + str(fleet_number).zfill(4)
        fleet_rows.append(bronze_row(
            'bronze_aircraft_fleet', aircraft_instance_id, BASE, aircraft_instance_id=aircraft_instance_id,
            tail_token='SYNTH-TAIL-' + str(fleet_number).zfill(4), aircraft_type_id=aircraft_type_id,
            airline_id=airline_id, base_airport_id=base_airport_id, fleet_status='Active',
            age_years=2 + fleet_number % 11, cumulative_flight_hours_proxy=2400 + fleet_number * 173,
            eligibility_id=assumption['eligibility_id']))

disciplines = ['Cleaning', 'Fueling', 'Catering', 'Baggage', 'Maintenance', 'Operations', 'Retail']
roles = {
    'Cleaning': 'Turnaround Specialist', 'Fueling': 'Fuel Operations Specialist',
    'Catering': 'Catering Coordinator', 'Baggage': 'Baggage Operations Specialist',
    'Maintenance': 'Maintenance Technician', 'Operations': 'Airport Operations Coordinator',
    'Retail': 'Commercial Operations Associate'}
skill_rows = [bronze_row(
    'bronze_skill', 'SKILL-' + discipline.upper(), BASE,
    skill_id='SKILL-' + discipline.upper(), skill_name=discipline + ' Operations',
    skill_category=discipline, certification_required=discipline in {'Fueling','Maintenance','Operations'})
    for discipline in disciplines]
shift_rows = [bronze_row(
    'bronze_shift', 'SHIFT-' + shift_name.upper(), BASE,
    shift_id='SHIFT-' + shift_name.upper(), shift_name=shift_name,
    start_hour_utc=start_hour, duration_hours=8)
    for shift_name, start_hour in [('Early',0),('Day',8),('Late',16)]]
work_team_rows, employee_rows, employee_skill_rows, roster_rows = [], [], [], []
gates_by_airport = {airport['airport_id']: [gate for gate in gates if gate['airport_id'] == airport['airport_id']] for airport in airports}
for airport in airports:
    for discipline in disciplines:
        work_team_id = airport['airport_id'] + '-TEAM-' + discipline[:3].upper()
        work_team_rows.append(bronze_row(
            'bronze_work_team', work_team_id, BASE, work_team_id=work_team_id,
            airport_id=airport['airport_id'], discipline=discipline,
            team_type='MaintenanceTeam' if discipline == 'Maintenance' else 'ServiceTeam',
            team_label=discipline + ' Team ' + airport['airport_id']))
    airport_gates = gates_by_airport[airport['airport_id']]
    for employee_index in range(EMPLOYEES_PER_AIRPORT):
        discipline = disciplines[employee_index % len(disciplines)]
        employee_id = 'WRK-' + airport['airport_id'] + '-' + str(employee_index + 1).zfill(3)
        team_id = airport['airport_id'] + '-TEAM-' + discipline[:3].upper()
        employee_rows.append(bronze_row(
            'bronze_employee', employee_id, BASE, employee_id=employee_id,
            home_airport_id=airport['airport_id'], work_team_id=team_id, discipline=discipline,
            role=roles[discipline], employment_type='SyntheticRosterMember', training_status='Current',
            identity_classification='PseudonymousSyntheticWorkforceToken'))
        employee_skill_rows.append(bronze_row(
            'bronze_employee_skill', employee_id + '-' + discipline, BASE,
            employee_skill_id=employee_id + '-' + discipline, employee_id=employee_id,
            skill_id='SKILL-' + discipline.upper(), proficiency_level=2 + employee_index % 3))
        shift_number = employee_index % 3
        shift_name = ['Early','Day','Late'][shift_number]
        shift_start = BASE + timedelta(hours=shift_number * 8)
        gate = airport_gates[employee_index % len(airport_gates)]
        planned_hours = 8.0
        actual_hours = round(7.5 + ((employee_index * 7 + int(config['random_seed'])) % 17) / 10.0, 1)
        roster_id = 'ROSTER-' + employee_id
        roster_rows.append(bronze_row(
            'bronze_employee_roster', roster_id, shift_start, roster_assignment_id=roster_id,
            employee_id=employee_id, airport_id=airport['airport_id'], work_team_id=team_id,
            assigned_terminal=gate['terminal'], assigned_gate_id=gate['gate_id'],
            shift_id='SHIFT-' + shift_name.upper(), shift_name=shift_name,
            shift_start=shift_start, shift_end=shift_start + timedelta(hours=actual_hours),
            planned_hours=planned_hours, actual_hours=actual_hours,
            overtime_hours=max(0.0, round(actual_hours - planned_hours, 1))))

terminal_keys = sorted({(gate['airport_id'], gate['terminal']) for gate in gates})
outlet_categories = ['FoodAndBeverage','Convenience','TravelEssentials','DutyFreeProxy']
retail_outlet_rows = []
for airport_id, terminal_code in terminal_keys:
    for outlet_index in range(OUTLETS_PER_TERMINAL):
        outlet_id = 'RTL-' + airport_id + '-' + terminal_code + '-' + str(outlet_index + 1).zfill(2)
        retail_outlet_rows.append(bronze_row(
            'bronze_retail_outlet', outlet_id, BASE, outlet_id=outlet_id, airport_id=airport_id,
            terminal_id=airport_id + '-' + terminal_code, outlet_name='Fictional Outlet ' + str(outlet_index + 1),
            outlet_category=outlet_categories[outlet_index % len(outlet_categories)],
            concession_model='SyntheticRevenueShare'))

In [ ]:
# Flight routing, airline-consistent fleet assignment, rotations, passengers/bookings, baggage, and customer experience.
route_records = [row.asDict() for row in route_rows]
fleet_records = [row.asDict() for row in fleet_rows]
routes_by_origin_airline = {}
for route in route_records:
    routes_by_origin_airline.setdefault((route['origin_airport_id'], route['airline_id']), []).append(route)
fleet_by_pair = {}
for aircraft_instance in fleet_records:
    pair = (aircraft_instance['aircraft_type_id'], aircraft_instance['airline_id'])
    fleet_by_pair.setdefault(pair, []).append(aircraft_instance)

fleet_available_at = {record['aircraft_instance_id']: BASE - timedelta(days=1) for record in fleet_records}
reserve_sequence = 0
flight_route_rows, rotation_rows = [], []
passenger_rows, booking_rows, baggage_rows, customer_experience_rows = [], [], [], []
segments = ['Business', 'Leisure', 'VisitingFriendsAndRelatives']
fare_classes = ['Economy', 'PremiumEconomy', 'Business']
booking_channels = ['DirectWeb', 'Mobile', 'TravelPartner', 'CorporatePortal']
assigned_flights_by_aircraft = {}
ordered_flights = sorted(flights, key=lambda item: (item['scheduled_arrival'], item['flight_event_id']))

for flight_index, flight in enumerate(ordered_flights):
    route_key = (flight['airport_id'], flight['airline_id'])
    if route_key not in routes_by_origin_airline:
        origin = next(airport for airport in airports if airport['airport_id'] == flight['airport_id'])
        destinations = [airport for airport in airports if airport['airport_id'] != origin['airport_id']]
        destination = destinations[(flight_index + sum(ord(char) for char in flight['airline_id'])) % len(destinations)]
        route_id = 'R-' + origin['airport_id'] + '-AIRLINE-' + flight['airline_id']
        fallback = bronze_row(
            'bronze_route', route_id, BASE, route_id=route_id, origin_airport_id=origin['airport_id'],
            destination_airport_id=destination['airport_id'], airline_id=flight['airline_id'],
            route_category='InterRegion' if origin['region'] != destination['region'] else 'IntraRegion',
            distance_km=distance_km(origin, destination), scheduled_frequency_daily=1,
            service_assumption_id='SYN-SERVICE-' + origin['airport_id'] + '-' + flight['airline_id'])
        route_rows.append(fallback)
        fallback_record = fallback.asDict()
        route_records.append(fallback_record)
        routes_by_origin_airline[route_key] = [fallback_record]
    route_candidates = routes_by_origin_airline[route_key]
    route = route_candidates[flight_index % len(route_candidates)]
    pair = (flight['aircraft_type_id'], flight['airline_id'])
    fleet_candidates = fleet_by_pair[pair]
    available_candidates = [
        candidate for candidate in fleet_candidates
        if fleet_available_at[candidate['aircraft_instance_id']] <= flight['scheduled_arrival']]
    if not available_candidates:
        reserve_sequence += 1
        aircraft_instance_id = 'FLEET-RESERVE-' + str(reserve_sequence).zfill(4)
        reserve_row = bronze_row(
            'bronze_aircraft_fleet', aircraft_instance_id, BASE,
            aircraft_instance_id=aircraft_instance_id, tail_token='SYNTH-RESERVE-' + str(reserve_sequence).zfill(4),
            aircraft_type_id=flight['aircraft_type_id'], airline_id=flight['airline_id'],
            base_airport_id=flight['airport_id'], fleet_status='ActiveReserve', age_years=1 + reserve_sequence % 8,
            cumulative_flight_hours_proxy=1200 + reserve_sequence * 97,
            eligibility_id='ELIG-' + flight['airline_id'] + '-' + flight['aircraft_type_id'])
        fleet_rows.append(reserve_row)
        reserve_record = reserve_row.asDict()
        fleet_records.append(reserve_record)
        fleet_by_pair[pair].append(reserve_record)
        fleet_available_at[aircraft_instance_id] = BASE - timedelta(days=1)
        available_candidates = [reserve_record]
    aircraft_instance = min(
        available_candidates,
        key=lambda candidate: (fleet_available_at[candidate['aircraft_instance_id']], candidate['aircraft_instance_id']))
    aircraft_instance_id = aircraft_instance['aircraft_instance_id']
    fleet_available_at[aircraft_instance_id] = flight['actual_departure']
    assigned_flights_by_aircraft.setdefault(aircraft_instance_id, []).append(flight)

    flight_route_rows.append(bronze_row(
        'bronze_flight_route', flight['flight_event_id'], flight['scheduled_arrival'],
        flight_event_id=flight['flight_event_id'], route_id=route['route_id'],
        destination_airport_id=route['destination_airport_id'], aircraft_instance_id=aircraft_instance_id))

    segment_counts = {segment: 0 for segment in segments}
    satisfaction_totals = {segment: 0.0 for segment in segments}
    for passenger_index in range(int(flight['passenger_count'])):
        passenger_token = stable_token('passenger', flight['flight_event_id'], passenger_index)
        booking_id = 'BK-' + stable_token('booking', flight['flight_event_id'], passenger_index)[:16].upper()
        segment = segments[(passenger_index + flight_index) % len(segments)]
        fare_class = RNG.choices(fare_classes, weights=[82, 12, 6], k=1)[0]
        booking_channel = booking_channels[(passenger_index + flight_index) % len(booking_channels)]
        booked_days_ahead = RNG.randint(2, 90)
        checked_bag_count = RNG.choices([0, 1, 2], weights=[30, 62, 8], k=1)[0]
        revenue_base = {'Economy': 145.0, 'PremiumEconomy': 245.0, 'Business': 520.0}[fare_class]
        ticket_revenue = round(revenue_base + route['distance_km'] * 0.08 + RNG.uniform(-18.0, 24.0), 2)
        assistance_flag = passenger_index % 31 == 0
        passenger_rows.append(bronze_row(
            'bronze_passenger', passenger_token, flight['scheduled_arrival'],
            passenger_token=passenger_token, customer_segment=segment,
            assistance_required=assistance_flag, identity_classification='PseudonymousSyntheticToken'))
        booking_rows.append(bronze_row(
            'bronze_booking', booking_id, flight['scheduled_arrival'] - timedelta(days=booked_days_ahead),
            booking_id=booking_id, passenger_token=passenger_token, flight_event_id=flight['flight_event_id'],
            route_id=route['route_id'], customer_segment=segment, fare_class=fare_class,
            booking_channel=booking_channel, booked_days_ahead=booked_days_ahead,
            ticket_revenue_proxy=ticket_revenue, checked_bag_count=checked_bag_count, booking_status='Flown'))
        for bag_index in range(checked_bag_count):
            bag_token = stable_token('bag', booking_id, bag_index)
            mishandled = (passenger_index + bag_index + flight_index) % 137 == 0
            loaded_timestamp = flight['turnaround_start'] + timedelta(minutes=9 + bag_index * 2)
            reclaim_timestamp = flight['actual_departure'] + timedelta(minutes=70 + (passenger_index % 18))
            baggage_rows.append(bronze_row(
                'bronze_baggage_journey', bag_token, loaded_timestamp,
                bag_token=bag_token, booking_id=booking_id, flight_event_id=flight['flight_event_id'],
                origin_airport_id=flight['airport_id'], destination_airport_id=route['destination_airport_id'],
                loaded_timestamp=loaded_timestamp, reclaim_timestamp=reclaim_timestamp,
                journey_status='Exception' if mishandled else 'Delivered',
                mishandled_flag=bool(mishandled), scan_count=3 if mishandled else 4))
        delay_penalty = min(2.0, max(0.0, (flight['actual_departure'] - flight['scheduled_departure']).total_seconds() / 3600.0))
        satisfaction = round(max(1.0, min(5.0, 4.5 - delay_penalty + RNG.uniform(-0.35, 0.35))), 2)
        segment_counts[segment] += 1
        satisfaction_totals[segment] += satisfaction

    for segment_index, segment in enumerate(segments):
        respondents = max(1, segment_counts[segment] // 4)
        average_satisfaction = round(satisfaction_totals[segment] / max(1, segment_counts[segment]), 2)
        nps_proxy = int(round((average_satisfaction - 3.0) * 35))
        cx_event_id = 'CX-' + flight['flight_event_id'] + '-' + str(segment_index + 1)
        customer_experience_rows.append(bronze_row(
            'bronze_customer_experience', cx_event_id, flight['actual_departure'],
            cx_event_id=cx_event_id, flight_event_id=flight['flight_event_id'], route_id=route['route_id'],
            airport_id=flight['airport_id'], customer_segment=segment,
            respondent_count=respondents, satisfaction_score=average_satisfaction,
            nps_proxy=max(-100, min(100, nps_proxy))))

for aircraft_instance_id, assigned_flights in assigned_flights_by_aircraft.items():
    previous_flight = None
    for sequence, flight in enumerate(sorted(assigned_flights, key=lambda item: item['scheduled_arrival']), start=1):
        ground_interval_min = None if previous_flight is None else round(
            (flight['scheduled_arrival'] - previous_flight['actual_departure']).total_seconds() / 60.0, 1)
        rotation_id = 'ROT-' + aircraft_instance_id + '-' + str(sequence).zfill(3)
        rotation_rows.append(bronze_row(
            'bronze_aircraft_rotation', rotation_id, flight['scheduled_arrival'],
            rotation_id=rotation_id, aircraft_instance_id=aircraft_instance_id,
            rotation_sequence=sequence, flight_event_id=flight['flight_event_id'],
            previous_flight_event_id=None if previous_flight is None else previous_flight['flight_event_id'],
            ground_interval_min=ground_interval_min, overlap_flag=False))
        previous_flight = flight

In [ ]:
# Additional operational domains: customers, flight legs, boarding, bag scans, ramp tasks, work orders, and recommendations.
flight_by_id = {flight['flight_event_id']: flight for flight in flights}
route_by_id = {route['route_id']: route for route in route_records}
customer_rows, flight_leg_rows, boarding_rows, baggage_scan_rows = [], [], [], []
ramp_task_rows, work_order_rows, recommendation_rows = [], [], []

for passenger in passenger_rows:
    record = passenger.asDict()
    customer_token = stable_token('customer', record['passenger_token'])
    customer_rows.append(bronze_row(
        'bronze_customer', customer_token, record['source_event_timestamp'],
        customer_token=customer_token, passenger_token=record['passenger_token'],
        customer_segment=record['customer_segment'], profile_classification='PseudonymousSyntheticProfile'))

for flight_route in flight_route_rows:
    record = flight_route.asDict()
    flight = flight_by_id[record['flight_event_id']]
    route = route_by_id[record['route_id']]
    leg_id = 'LEG-' + record['flight_event_id']
    flight_leg_rows.append(bronze_row(
        'bronze_flight_leg', leg_id, flight['scheduled_arrival'], leg_id=leg_id,
        flight_event_id=record['flight_event_id'], route_id=record['route_id'],
        origin_airport_id=flight['airport_id'], destination_airport_id=route['destination_airport_id'],
        scheduled_departure_utc=flight['scheduled_departure'], actual_departure_utc=flight['actual_departure'],
        scheduled_arrival_utc=flight['scheduled_arrival'], actual_arrival_utc=flight['actual_arrival']))

for booking in booking_rows:
    record = booking.asDict()
    flight = flight_by_id[record['flight_event_id']]
    boarding_time = flight['actual_departure'] - timedelta(minutes=25 - (int(record['booking_id'][-2:], 16) % 12))
    boarding_rows.append(bronze_row(
        'bronze_boarding_event', 'BOARD-' + record['booking_id'], boarding_time,
        boarding_event_id='BOARD-' + record['booking_id'], booking_id=record['booking_id'],
        passenger_token=record['passenger_token'], flight_event_id=record['flight_event_id'],
        gate_id=flight['gate_id'], boarding_status='Boarded', boarding_timestamp_utc=boarding_time,
        boarding_window_risk='Watch' if (flight['actual_departure'] - boarding_time).total_seconds() / 60 < 18 else 'Normal'))

for baggage in baggage_rows:
    record = baggage.asDict()
    scan_stages = [
        ('Accepted', record['loaded_timestamp'] - timedelta(minutes=35)),
        ('Sorted', record['loaded_timestamp'] - timedelta(minutes=15)),
        ('Loaded', record['loaded_timestamp']),
        ('Reclaim', record['reclaim_timestamp'])]
    if record['mishandled_flag']:
        scan_stages = scan_stages[:-1]
    for sequence, (stage, scan_time) in enumerate(scan_stages, start=1):
        scan_id = 'SCAN-' + record['bag_token'] + '-' + str(sequence)
        baggage_scan_rows.append(bronze_row(
            'bronze_baggage_scan', scan_id, scan_time, baggage_scan_id=scan_id,
            bag_token=record['bag_token'], flight_event_id=record['flight_event_id'],
            scan_sequence=sequence, scan_stage=stage, scan_timestamp_utc=scan_time))

ramp_phases = ['ChocksAndSafety','BaggageUnload','CleaningAndCatering','FuelAndWater','BoardingClose']
for flight in flights:
    for task_sequence, task_name in enumerate(ramp_phases, start=1):
        task_id = 'RAMP-' + flight['flight_event_id'] + '-' + str(task_sequence)
        planned_start = flight['turnaround_start'] + timedelta(minutes=(task_sequence - 1) * 6)
        variance = ((sum(ord(char) for char in task_id) + int(config['random_seed'])) % 9) - 2
        actual_end = planned_start + timedelta(minutes=8 + task_sequence + variance)
        ramp_task_rows.append(bronze_row(
            'bronze_ramp_service_task', task_id, actual_end, ramp_task_id=task_id,
            flight_event_id=flight['flight_event_id'], airport_id=flight['airport_id'], gate_id=flight['gate_id'],
            task_name=task_name, task_sequence=task_sequence, planned_start_utc=planned_start,
            actual_end_utc=actual_end, task_status='Late' if variance > 4 else 'Completed'))

for maintenance in rows('bronze_maintenance'):
    work_order_id = 'WO-' + maintenance['maintenance_id']
    resolution_hours = 2 + (sum(ord(char) for char in work_order_id) % 22)
    work_order_rows.append(bronze_row(
        'bronze_maintenance_work_order', work_order_id, maintenance['event_time'],
        work_order_id=work_order_id, maintenance_id=maintenance['maintenance_id'],
        airport_id=maintenance['airport_id'], gate_id=maintenance['gate_id'],
        asset_type=maintenance['asset_type'], team_id=maintenance['team_id'],
        work_order_type='Preventive' if int(maintenance['maintenance_id'][-2:]) % 3 else 'Corrective',
        opened_at_utc=maintenance['event_time'], resolved_at_utc=maintenance['event_time'] + timedelta(hours=resolution_hours),
        resolution_hours=float(resolution_hours), status=maintenance['status'], approval_status='ApprovedForDemoAnalysis'))

for airport_index, airport in enumerate(airports):
    recommendation_id = 'REC-' + airport['airport_id']
    recommendation_rows.append(bronze_row(
        'bronze_recommendation_event', recommendation_id, OBSERVATION_TS,
        recommendation_id=recommendation_id, airport_id=airport['airport_id'],
        recommendation_type='StaffingReview' if airport_index % 2 else 'TurnaroundReview',
        recommendation_text='Authorized manager should review the synthetic analytical evidence',
        recommendation_status='PendingHumanReview' if airport_index % 3 == 0 else 'AcceptedForScenario',
        approval_required=True, approved_by_token=None, advisory_only=True))

In [ ]:
# Inventory snapshots and asset inspections.
inventory_rows = []
for outlet_index, outlet_row in enumerate(retail_outlet_rows):
    outlet = outlet_row.asDict()
    for product_index in range(12):
        product_id = 'PRD-' + str(product_index + 1).zfill(3)
        inventory_id = 'INV-' + outlet['outlet_id'] + '-' + product_id
        on_hand_units = 8 + ((outlet_index * 13 + product_index * 7 + int(config['random_seed'])) % 48)
        reorder_point = 12 + product_index % 6
        inventory_rows.append(bronze_row(
            'bronze_retail_inventory', inventory_id, OBSERVATION_TS,
            inventory_snapshot_id=inventory_id, outlet_id=outlet['outlet_id'], product_id=product_id,
            airport_id=outlet['airport_id'], terminal_id=outlet['terminal_id'],
            on_hand_units=on_hand_units, reorder_point_units=reorder_point,
            stock_status='Reorder' if on_hand_units <= reorder_point else 'Available'))

inspection_rows = []
for asset_index, asset in enumerate(sorted(rows('bronze_asset_registry'), key=lambda row: row['asset_id'])):
    inspection_id = 'INSP-' + asset['asset_id']
    score = 72 + ((asset_index * 11 + int(config['random_seed'])) % 29)
    inspection_time = BASE + timedelta(hours=asset_index % int(config['simulation_hours']))
    inspection_rows.append(bronze_row(
        'bronze_asset_inspection', inspection_id, inspection_time,
        inspection_id=inspection_id, asset_id=asset['asset_id'], airport_id=asset['airport_id'],
        gate_id=asset['gate_id'], inspection_type='SyntheticConditionInspection',
        inspection_score=score, inspection_status='Action' if score < 80 else ('Watch' if score < 90 else 'Pass'),
        inspected_at_utc=inspection_time, follow_up_required=score < 90))

ENTERPRISE_KEYS.update({
    'bronze_aircraft_rotation':'rotation_id',
    'bronze_retail_inventory':'inventory_snapshot_id',
    'bronze_asset_inspection':'inspection_id'})

In [ ]:
# Retail/POS, turnaround milestone events, and isolated quality test cases.
product_categories = ['FoodAndBeverage', 'Convenience', 'TravelEssentials', 'DutyFreeProxy']
retail_product_rows = []
for product_index in range(12):
    product_id = 'PRD-' + str(product_index + 1).zfill(3)
    retail_product_rows.append(bronze_row(
        'bronze_retail_product', product_id, BASE,
        product_id=product_id, product_name='Fictional Product ' + str(product_index + 1),
        product_category=product_categories[product_index % len(product_categories)],
        unit_price_proxy=round(4.5 + product_index * 3.25, 2),
    ))

product_records = [row.asDict() for row in retail_product_rows]
RETAIL_CONVERSION_PROPENSITY = float(config['retail_conversion_propensity'])
passengers_by_airport_hour = {}
for flight in flights:
    arrival_hour = int((flight['scheduled_arrival'] - BASE).total_seconds() // 3600)
    passenger_key = (flight['airport_id'], arrival_hour)
    passengers_by_airport_hour[passenger_key] = (passengers_by_airport_hour.get(passenger_key, 0) + int(flight['passenger_count']))
outlets_by_airport = {}
for outlet_row in retail_outlet_rows:
    outlet_airport_id = outlet_row.asDict()['airport_id']
    outlets_by_airport[outlet_airport_id] = outlets_by_airport.get(outlet_airport_id, 0) + 1

retail_pos_rows = []
for outlet_index, outlet_row in enumerate(retail_outlet_rows):
    outlet = outlet_row.asDict()
    matching_products = [product for product in product_records if product['product_category'] == outlet['outlet_category']]
    if not matching_products:
        matching_products = product_records
    for hour in range(int(config['simulation_hours'])):
        event_timestamp = BASE + timedelta(hours=hour)
        product = matching_products[(outlet_index + hour) % len(matching_products)]
        demand_multiplier = 1.45 if hour % 24 in [7, 8, 12, 13, 17, 18] else 0.82
        hourly_passengers = passengers_by_airport_hour.get((outlet['airport_id'], hour), 0)
        outlet_passenger_share = (hourly_passengers * RETAIL_CONVERSION_PROPENSITY
                                  / max(1, outlets_by_airport[outlet['airport_id']]))
        transaction_count = max(0, int(round(outlet_passenger_share * demand_multiplier)))
        average_basket = round(product['unit_price_proxy'] * (1.15 + RNG.uniform(0.0, 1.4)), 2)
        gross_sales = round(transaction_count * average_basket, 2)
        refund = round(gross_sales * (0.01 if (hour + outlet_index) % 11 else 0.035), 2)
        pos_event_id = 'POS-' + outlet['outlet_id'] + '-' + str(hour).zfill(2)
        retail_pos_rows.append(bronze_row(
            'bronze_retail_pos', pos_event_id, event_timestamp,
            pos_event_id=pos_event_id, outlet_id=outlet['outlet_id'], product_id=product['product_id'],
            airport_id=outlet['airport_id'], terminal_id=outlet['terminal_id'], event_hour=hour,
            transaction_count=transaction_count, gross_sales_proxy=gross_sales,
            refund_proxy=refund, average_basket_proxy=average_basket,
        ))

phase_definitions = [
    ('Deboarding', 0.16), ('BaggageOffload', 0.18), ('CleaningAndCatering', 0.27),
    ('Boarding', 0.29), ('PushbackReady', 0.10),
]
turnaround_phase_rows = []
for flight in flights:
    total_seconds = max(300, int((flight['turnaround_end'] - flight['turnaround_start']).total_seconds()))
    phase_start = flight['turnaround_start']
    for phase_index, (phase_name, phase_weight) in enumerate(phase_definitions):
        if phase_index == len(phase_definitions) - 1:
            phase_end = flight['turnaround_end']
        else:
            phase_end = phase_start + timedelta(seconds=int(total_seconds * phase_weight))
        phase_event_id = 'PH-' + flight['flight_event_id'] + '-' + str(phase_index + 1)
        phase_duration = round((phase_end - phase_start).total_seconds() / 60.0, 2)
        turnaround_phase_rows.append(bronze_row(
            'bronze_turnaround_phase', phase_event_id, phase_end,
            phase_event_id=phase_event_id, flight_event_id=flight['flight_event_id'],
            airport_id=flight['airport_id'], gate_id=flight['gate_id'], phase_name=phase_name,
            phase_sequence=phase_index + 1, phase_start=phase_start, phase_end=phase_end,
            phase_duration_min=phase_duration,
            milestone_status='Delayed' if phase_duration > total_seconds / 60.0 * phase_weight * 1.1 else 'OnPlan',
        ))
        phase_start = phase_end

quality_case_rows = []
quality_cases = ['LateArrival', 'DuplicateEvent', 'MalformedPayload', 'OutOfOrderEvent']
for case_index in range(12):
    test_case_id = 'QUALITY-CASE-' + str(case_index + 1).zfill(3)
    case_type = quality_cases[case_index % len(quality_cases)]
    quality_case_rows.append(bronze_row(
        'bronze_event_quality_cases', test_case_id, BASE + timedelta(hours=case_index),
        test_case_id=test_case_id, case_type=case_type,
        synthetic_payload='{' + '"case":"' + case_type + '","synthetic":true' + '}',
        expected_disposition='Quarantine', test_only=True,
    ))

In [ ]:
table_rows = {
    'bronze_organization':organization_rows,'bronze_route':route_rows,
    'bronze_aircraft_fleet':fleet_rows,'bronze_aircraft_rotation':rotation_rows,
    'bronze_work_team':work_team_rows,'bronze_skill':skill_rows,'bronze_shift':shift_rows,
    'bronze_employee':employee_rows,'bronze_employee_skill':employee_skill_rows,
    'bronze_employee_roster':roster_rows,'bronze_retail_outlet':retail_outlet_rows,
    'bronze_retail_product':retail_product_rows,'bronze_retail_inventory':inventory_rows,
    'bronze_flight_route':flight_route_rows,'bronze_flight_leg':flight_leg_rows,
    'bronze_customer':customer_rows,'bronze_passenger':passenger_rows,'bronze_booking':booking_rows,
    'bronze_boarding_event':boarding_rows,'bronze_baggage_journey':baggage_rows,
    'bronze_baggage_scan':baggage_scan_rows,'bronze_ramp_service_task':ramp_task_rows,
    'bronze_maintenance_work_order':work_order_rows,'bronze_asset_inspection':inspection_rows,
    'bronze_retail_pos':retail_pos_rows,'bronze_turnaround_phase':turnaround_phase_rows,
    'bronze_customer_experience':customer_experience_rows,
    'bronze_recommendation_event':recommendation_rows,'bronze_event_quality_cases':quality_case_rows}

for table_name, generated_rows in table_rows.items():
    write_delta(generated_rows, table_name)

required_metadata = {
    'source_event_timestamp','ingestion_timestamp','source_record_key','schema_version','payload_hash',
    'batch_id','generated_at_utc','generator_version','random_seed','record_source','data_classification',
    'source_name','source_url','source_as_of_date','is_synthetic'}
summary_frames = []
for table_name, primary_key in ENTERPRISE_KEYS.items():
    frame = spark.table(table_name)
    assert required_metadata.issubset(set(frame.columns)), table_name + ' missing metadata'
    summary_frames.append(frame.agg(
        F.count(F.lit(1)).alias('row_count'),
        F.countDistinct(primary_key).alias('distinct_key_count'),
        F.sum(F.when(F.col(primary_key).isNull(), 1).otherwise(0)).alias('null_key_count'),
        F.sum(F.when(F.col('is_synthetic').isNull() | ~F.col('is_synthetic'), 1).otherwise(0)).alias('invalid_synthetic_count'),
    ).withColumn('table_name', F.lit(table_name)))

summary = summary_frames[0]
for summary_frame in summary_frames[1:]:
    summary = summary.unionByName(summary_frame)
summary_by_table = {row['table_name']: row.asDict() for row in summary.collect()}
for table_name, generated_rows in table_rows.items():
    metrics = summary_by_table[table_name]
    assert metrics['row_count'] == len(generated_rows) > 0, table_name + ' row count mismatch'
    assert metrics['row_count'] == metrics['distinct_key_count'], table_name + ' duplicate primary keys'
    assert metrics['null_key_count'] == 0, table_name + ' null primary keys'
    assert metrics['invalid_synthetic_count'] == 0, table_name + ' invalid synthetic records'

orphan_checks = [
    ('bronze_employee_skill','employee_id','bronze_employee','employee_id'),
    ('bronze_employee_skill','skill_id','bronze_skill','skill_id'),
    ('bronze_employee_roster','employee_id','bronze_employee','employee_id'),
    ('bronze_employee_roster','work_team_id','bronze_work_team','work_team_id'),
    ('bronze_flight_route','flight_event_id','bronze_flight_turnaround','flight_event_id'),
    ('bronze_aircraft_rotation','aircraft_instance_id','bronze_aircraft_fleet','aircraft_instance_id'),
    ('bronze_aircraft_rotation','flight_event_id','bronze_flight_turnaround','flight_event_id'),
    ('bronze_flight_leg','flight_event_id','bronze_flight_turnaround','flight_event_id'),
    ('bronze_customer','passenger_token','bronze_passenger','passenger_token'),
    ('bronze_booking','passenger_token','bronze_passenger','passenger_token'),
    ('bronze_booking','route_id','bronze_route','route_id'),
    ('bronze_boarding_event','booking_id','bronze_booking','booking_id'),
    ('bronze_baggage_journey','booking_id','bronze_booking','booking_id'),
    ('bronze_baggage_scan','bag_token','bronze_baggage_journey','bag_token'),
    ('bronze_ramp_service_task','flight_event_id','bronze_flight_turnaround','flight_event_id'),
    ('bronze_maintenance_work_order','maintenance_id','bronze_maintenance','maintenance_id'),
    ('bronze_asset_inspection','asset_id','bronze_asset_registry','asset_id'),
    ('bronze_retail_inventory','outlet_id','bronze_retail_outlet','outlet_id'),
    ('bronze_retail_inventory','product_id','bronze_retail_product','product_id'),
    ('bronze_retail_pos','outlet_id','bronze_retail_outlet','outlet_id'),
    ('bronze_retail_pos','product_id','bronze_retail_product','product_id'),
    ('bronze_turnaround_phase','flight_event_id','bronze_flight_turnaround','flight_event_id'),
    ('bronze_customer_experience','flight_event_id','bronze_flight_turnaround','flight_event_id')]
orphan_frames = []
for child_table, child_key, parent_table, parent_key in orphan_checks:
    rule = child_table + '.' + child_key + ' -> ' + parent_table + '.' + parent_key
    orphan_frames.append(
        spark.table(child_table).select(child_key).distinct()
        .join(spark.table(parent_table).select(F.col(parent_key).alias(child_key)).distinct(), child_key, 'left_anti')
        .limit(1).select(F.lit(rule).alias('rule')))
orphan_summary = orphan_frames[0]
for orphan_frame in orphan_frames[1:]:
    orphan_summary = orphan_summary.unionByName(orphan_frame)
orphan_violations = [row['rule'] for row in orphan_summary.collect()]
assert not orphan_violations, 'Orphan relationships: ' + ', '.join(orphan_violations)

forbidden_columns = {'name','email','phone','address','passport','biometric','credential','tenant_id','payment'}
for sensitive_table in ['bronze_passenger','bronze_customer','bronze_booking','bronze_employee']:
    assert not forbidden_columns.intersection({column.lower() for column in spark.table(sensitive_table).columns})

assert summary_by_table['bronze_aircraft_rotation']['row_count'] == len(flights)
assert spark.table('bronze_aircraft_rotation').filter(F.col('overlap_flag')).limit(1).count() == 0
service_pairs = {(assumption['airport_id'], assumption['airline_id']) for assumption in service_assumptions}
eligibility_pairs = {(assumption['airline_id'], assumption['aircraft_type_id']) for assumption in eligibility_assumptions}
assert all((route['origin_airport_id'],route['airline_id']) in service_pairs for route in route_records)
assert all((fleet['airline_id'],fleet['aircraft_type_id']) in eligibility_pairs for fleet in fleet_records)
print('PASS: enterprise Bronze rotations, inventory, inspections, metadata, eligibility, pseudonymization, uniqueness, and integrity')